# Практична робота №3 (оновлено)
## Класифікація твітів на основі **реального датасету** з файлу *Corona_NLP_test.csv.xls*

- Завантаження даних **із файлу** (без сторонніх CSV та без `pip install`).
- Очищення тексту → токенізація → стоп-слова.
- Векторизація Word2Vec (усереднення векторів слів для твіта).
- Класифікація: Logistic Regression, Linear SVM, Random Forest, Gaussian NB.
- Порівняння **без PCA** та з **PCA(50/100/200)** (автоматично обрізається, якщо вибірка замала).
- Метрики: accuracy + матриця плутанини.
- Прогноз для нового тексту.

In [ ]:
import os, re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [ ]:
CANDIDATES = [
    "Corona_NLP_test.csv.xls",
    "Corona_NLP_test.csv",
    "Corona_NLP_test.xlsx",
    "Corona_NLP_test.xls"
]

path = None
for name in CANDIDATES:
    if os.path.exists(name):
        path = name
        break
    if os.path.exists(os.path.join("/content", name)):
        path = os.path.join("/content", name)
        break

if path is None and IN_COLAB:
    up = files.upload()
    if len(up):
        path = next(iter(up.keys()))

if path is None:
    raise FileNotFoundError("File not found.")

df = None
csv_err = None
for enc in ("utf-8", "latin-1", "cp1252"):
    try:
        df = pd.read_csv(path, encoding=enc)
        break
    except Exception as e:
        csv_err = e
        df = None

if df is None:
    try:
        df = pd.read_excel(path)
    except Exception as e:
        raise RuntimeError("Re-save file as CSV or XLSX and upload again.")

print(df.shape)
print(df.columns.tolist())

In [ ]:
text_col = None
label_col = None

for c in df.columns:
    cl = c.strip().lower()
    if cl in ("originaltweet", "text", "tweet", "content"):
        text_col = c
    if cl in ("sentiment", "label", "target"):
        label_col = c

if text_col is None or label_col is None:
    raise ValueError("Required columns not found.")

df = df[[text_col, label_col]].rename(columns={text_col:"OriginalTweet", label_col:"Sentiment"}).dropna()

def normalize_sentiment(s):
    s = str(s).strip().lower()
    if "positive" in s:
        return "Positive"
    if "negative" in s:
        return "Negative"
    return "Other"

df["Sentiment3"] = df["Sentiment"].apply(normalize_sentiment)

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_and_tokenize(text: str):
    t = str(text).lower()
    t = re.sub(r"http\S+|www\.\S+|@\w+|#", " ", t)
    t = re.sub(r"[^a-z\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    toks = word_tokenize(t)
    toks = [w for w in toks if w not in stop_words and len(w) > 2]
    return toks

df["tokens"] = df["OriginalTweet"].apply(clean_and_tokenize)

In [ ]:
train_df, test_df = train_test_split(
    df[["tokens","Sentiment3"]],
    test_size=0.2,
    random_state=42,
    stratify=df["Sentiment3"]
)

In [ ]:
EMBEDDING_DIM = 200

from gensim.models import Word2Vec
w2v_model = Word2Vec(
    sentences=train_df["tokens"].tolist(),
    vector_size=EMBEDDING_DIM,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=20
)

import numpy as np
def tweet_vec(tokens, model, dim=EMBEDDING_DIM):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

X_train = np.vstack([tweet_vec(t, w2v_model, EMBEDDING_DIM) for t in train_df["tokens"]])
X_test  = np.vstack([tweet_vec(t, w2v_model, EMBEDDING_DIM) for t in test_df["tokens"]])

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(train_df["Sentiment3"].values)
y_test  = le.transform(test_df["Sentiment3"].values)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

def fit_eval(clf, Xtr, ytr, Xte, yte):
    clf.fit(Xtr, ytr)
    yp = clf.predict(Xte)
    acc = accuracy_score(yte, yp)
    rep = classification_report(yte, yp, target_names=le.classes_, output_dict=True, zero_division=0)
    cm  = confusion_matrix(yte, yp)
    return acc, rep, cm, yp

def get_clfs():
    return {
        "LogReg": LogisticRegression(max_iter=2000),
        "LinearSVC": LinearSVC(),
        "RF": RandomForestClassifier(n_estimators=200, random_state=42),
        "GNB": GaussianNB()
    }

max_pca = int(min(X_train.shape[0], X_train.shape[1]))
pca_list = [0] + [k for k in [50,100,200] if 0 < k <= max_pca]

results = []
best = {"acc": -1, "name": None, "k": 0, "clf": None, "cm": None, "report": None}

for k in pca_list:
    if k == 0:
        Xtr_k, Xte_k = X_train, X_test
    else:
        pca = PCA(n_components=k, random_state=42)
        Xtr_k = pca.fit_transform(X_train)
        Xte_k = pca.transform(X_test)

    for name, clf in get_clfs().items():
        acc, rep, cm, yp = fit_eval(clf, Xtr_k, y_train, Xte_k, y_test)
        results.append({"model":name, "pca_components":k, "accuracy":acc, "macro_f1":rep["macro avg"]["f1-score"]})
        if acc > best["acc"]:
            best = {"acc": acc, "name": name, "k": k, "clf": clf, "cm": cm, "report": rep}

import pandas as pd
results_df = pd.DataFrame(results).sort_values(by="accuracy", ascending=False).reset_index(drop=True)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
for m in results_df["model"].unique():
    sub = results_df[results_df["model"] == m]
    plt.plot(sub["pca_components"], sub["accuracy"], marker='o', label=m)

plt.xlabel("PCA components (0 = no PCA)")
plt.ylabel("Accuracy")
plt.title("Accuracy vs PCA")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
cm = best["cm"]
labels = list(le.classes_)

fig, ax = plt.subplots(figsize=(4,4))
im = ax.imshow(cm, interpolation='nearest')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix ({best['name']}, PCA={best['k']})")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.show()

In [ ]:
def predict_text(raw_text: str, model_name: str, pca_k: int):
    from sklearn.decomposition import PCA
    toks = clean_and_tokenize(raw_text)
    v = tweet_vec(toks, w2v_model, 200).reshape(1, -1)

    if pca_k and pca_k > 0:
        pca = PCA(n_components=pca_k, random_state=42)
        Xtr = pca.fit_transform(X_train)
        vx = pca.transform(v)
    else:
        Xtr = X_train
        vx = v

    clfs = {
        "LogReg": LogisticRegression(max_iter=2000),
        "LinearSVC": LinearSVC(),
        "RF": RandomForestClassifier(n_estimators=200, random_state=42),
        "GNB": GaussianNB()
    }
    clf = clfs[model_name]
    clf.fit(Xtr, y_train)
    pred = clf.predict(vx)[0]
    return le.inverse_transform([pred])[0]

demo = "I just got my vaccine today and feel much better now"
print("Best model:", best["name"], "PCA=", best["k"])
print("Text:", demo)
print("Pred:", predict_text(demo, best["name"], best["k"]))

**Підсумок:** Дані читаються з вашого файлу, мітки зведені до 3 класів, 
проведено навчання 4 моделей з/без PCA, побудовано графік та матрицю плутанини,
є функція прогнозу для нового тексту.